# House Stage 1 regression: fixed incumbency + recency WLS, rolling 2018–2022

The classic formula with two changes from unrestricted OLS combined: the `inc_signed` coefficient **fixed at `FIXED_INC`** (2.5), and the fit by **WLS with recency weights** (half-life `HALF_LIFE`, 6 years). They come from `01_reg_2018to2022_fixed_inc.ipynb` and `01_reg_2018to2022_wls.ipynb`. The same expanding-window backtest refits every model on prior elections only and scores the next one:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

The combined model is `run_wls(..., half_life=HALF_LIFE, fixed={"inc_signed": FIXED_INC})`. It is WLS of `adj_margin_2p_norm − FIXED_INC · inc_signed` on the other terms, with weights 2^(−(T − t)/H) computed and normalized (mean 1) from each window's own training rows. `FIXED_INC · inc_signed` is then added back to the predictions.

Each change is also run on its own, so the table shows what each one contributes: OLS, fixed incumbency only, recency WLS only, and both. Every model in a window uses the same training and evaluation rows, and the evaluation subsets (valid, `COMPETITIVE`) are the original notebook's. 2024 is never used (`USE_HOLDOUT = False`).

In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import evaluation, tracking
from experiments.house.run import run_ols, run_ols_fixed, run_wls

In [2]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [3]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
REQUIRE_COMPLETE = []   # extra columns a row must have; every model below uses FORMULA's columns, so the same rows
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

In [4]:
FIXED_TERM = "inc_signed"
FIXED_INC = 2.5   # imposed incumbency coefficient, points of D margin
HALF_LIFE = 6     # recency half-life, years: a year HALF_LIFE years older counts half

OLS_NAME = "ols"
FIXED_NAME = f"inc_fixed_{FIXED_INC:g}"
RECENCY_NAME = f"wls_recency_h{HALF_LIFE:g}"
COMBINED_NAME = f"wls_recency_h{HALF_LIFE:g}_inc_fixed_{FIXED_INC:g}"

In [5]:
# Every window: OLS (the base), each change on its own, and both together, all on the same rows
rolling = {}
for eval_year, train_years in ROLLING_WINDOWS.items():
    ols = run_ols(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=OLS_NAME,
        save=False,
    )
    fixed = run_ols_fixed(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        fixed={FIXED_TERM: FIXED_INC},
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=FIXED_NAME,
        save=False,
    )
    recency = run_wls(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        half_life=HALF_LIFE,
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=RECENCY_NAME,
        save=False,
    )
    combined = run_wls(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        half_life=HALF_LIFE,
        fixed={FIXED_TERM: FIXED_INC},
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name=COMBINED_NAME,
        save=False,
    )
    rolling[eval_year] = {"ols_vs_combined": [ols, fixed, recency, combined]}  # OLS first: the base of every delta

/Users/ElijahSong/Documents/Election26/experiments/house/run.py:144: UserWarning: 2 predictions outside +-100 (not clipped): (2018, 'NY', 13), (2018, 'NY', 15)
  return _fit_and_score(
/Users/ElijahSong/Documents/Election26/experiments/house/run.py:121: UserWarning: 2 predictions outside +-100 (not clipped): (2018, 'NY', 13), (2018, 'NY', 15)
  return _fit_and_score(


In [6]:
# what each model fits (the same in every window)
pd.DataFrame({run.config["name"]: {"weights": tracking.weights_text(run.config),
                                   "fit formula": run.config.get("fit_formula", run.config["formula"])}
              for run in rolling[next(iter(ROLLING_WINDOWS))]["ols_vs_combined"]}).T

,weights,fit formula
ols,ols,adj_margin_2p_norm ~ gen_margin_raw + partisan...
inc_fixed_2.5,ols,I(adj_margin_2p_norm - 2.5 * inc_signed) ~ gen...
wls_recency_h6,wls: half_life 6,adj_margin_2p_norm ~ gen_margin_raw + partisan...
wls_recency_h6_inc_fixed_2.5,wls: half_life 6,I(adj_margin_2p_norm - 2.5 * inc_signed) ~ gen...


## Results

For each section (ALL, EX_ANTE and EX_POST on the original notebook's definitions), there are two tables. The first is **by eval year**, each window on its own. The second is the **mean over 2018, 2020 and 2022**, each window weighted equally. Variants are ranked by mean MAE, with ties broken by RMSE.

- **Metrics:** `n_train`, `n`, MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D/R and misses.
- **Mean table extras:** `|bias|` (the mean of each window's |bias|, so opposite misses don't cancel) and `mae_<year>` for each window.
- **Δ columns:** variant − unrestricted OLS in the same window, on the same rows. ✓ (green) marks an improvement and ✗ (red) a regression: lower MAE, RMSE, |bias| and misses, higher accuracy and precision.

In [7]:
VARIANTS = 3  # show every model, ranked

for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean.droplevel("block")))

### ALL (`valid`): by eval year

### ALL (`valid`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,400,7.31,8.91,-1.03,2.59,94.89,96.35,94.22,20,5.11,7.72,7.93,6.27,-,-,-,-,-,-,-,-
inc_fixed_2.5,2680,400,5.69,7.31,-0.48,1.91,94.65,96.14,93.44,21,5.35,6.16,5.97,4.93,-1.62 ✓,-1.60 ✓,-0.24 ✗,-0.21 ✗,-0.78 ✗,+1 ✗,+0.24 ✗,-0.68 ✓
wls_recency_h6_inc_fixed_2.5,2680,400,5.70,7.30,-0.29,2.24,94.65,96.30,93.31,21,5.35,6.32,5.95,4.84,-1.60 ✓,-1.61 ✓,-0.24 ✗,-0.05 ✗,-0.90 ✗,+1 ✗,+0.24 ✗,-0.35 ✓
wls_recency_h6,2680,400,6.74,8.28,-0.96,2.69,94.81,96.35,94.05,21,5.19,7.38,7.39,5.45,-0.57 ✓,-0.63 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.10 ✗


### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): by eval year

### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,91,7.93,9.24,+0.91,4.18,82.90,88.14,83.16,16,17.10,9.47,7.95,6.36,-,-,-,-,-,-,-,-
inc_fixed_2.5,2680,91,5.28,6.56,+1.17,1.88,80.37,85.68,76.71,18,19.63,5.57,4.89,5.37,-2.65 ✓,-2.68 ✓,-2.53 ✗,-2.46 ✗,-6.45 ✗,+2 ✗,+2.53 ✗,-2.30 ✓
wls_recency_h6_inc_fixed_2.5,2680,91,5.36,6.65,+1.35,2.27,80.45,86.11,76.73,18,19.55,5.68,4.92,5.49,-2.56 ✓,-2.59 ✓,-2.45 ✗,-2.03 ✗,-6.43 ✗,+2 ✗,+2.45 ✗,-1.91 ✓
wls_recency_h6,2680,91,7.05,8.27,+1.05,3.65,82.90,88.14,83.16,16,17.10,8.70,6.98,5.46,-0.88 ✓,-0.97 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.53 ✓


### EX_POST_COMPETITIVE (`competitive_ex_post`): by eval year

### EX_POST_COMPETITIVE (`competitive_ex_post`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,60,10.56,12.20,+0.86,6.55,69.93,80.00,74.20,18,30.07,12.25,11.88,7.54,-,-,-,-,-,-,-,-
inc_fixed_2.5,2680,60,6.43,8.27,+1.97,2.74,68.90,73.51,67.60,19,31.10,8.05,5.62,5.63,-4.12 ✓,-3.93 ✓,-1.04 ✗,-6.49 ✗,-6.60 ✗,+0 ✗,+1.04 ✗,-3.81 ✓
wls_recency_h6_inc_fixed_2.5,2680,60,6.46,8.31,+2.14,3.10,69.05,73.88,67.74,19,30.95,8.22,5.56,5.60,-4.10 ✓,-3.89 ✓,-0.89 ✗,-6.12 ✗,-6.46 ✗,+0 ✗,+0.89 ✗,-3.45 ✓
wls_recency_h6,2680,60,9.50,11.07,+1.21,6.03,69.29,79.69,73.21,19,30.71,11.58,10.38,6.52,-1.06 ✓,-1.13 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.52 ✓


## Coefficients

Each model's coefficients by window, then averaged over the three. For the fixed models, `inc_signed` is the imposed `FIXED_INC` (it has no standard error), and the other terms are re-estimated around it, by OLS or by recency-weighted WLS.

In [8]:
# coefficients by window (rows: eval year and model, columns: terms); inc_signed is imposed for the fixed variants
pd.concat({eval_year: pd.DataFrame({run.config["name"]: run.coefficients["coef"] for run in blocks["ols_vs_combined"]}).T
           for eval_year, blocks in rolling.items()}, names=["eval_year", "model"]).round(3)

term                                    Intercept  gen_margin_raw  \
eval_year model                                                     
2018      ols                              -0.179           0.875   
          inc_fixed_2.5                    -0.212           0.789   
          wls_recency_h6                   -0.341           0.911   
          wls_recency_h6_inc_fixed_2.5     -0.666           0.828   
2020      ols                              -0.021           0.892   
          inc_fixed_2.5                    -0.061           0.811   
          wls_recency_h6                   -0.203           0.955   
          wls_recency_h6_inc_fixed_2.5     -0.450           0.893   
2022      ols                              -0.483           0.827   
          inc_fixed_2.5                    -0.372           0.772   
          wls_recency_h6                   -0.946           0.800   
          wls_recency_h6_inc_fixed_2.5     -0.960           0.786   

term                                    partisan_lean  inc_signed  
eval_year model                                                    
2018      ols                                   0.807      13.811  
          inc_fixed_2.5                         1.067       2.500  
          wls_recency_h6                        0.840      12.527  
          wls_recency_h6_inc_fixed_2.5          1.074       2.500  
2020      ols                                   0.809      12.798  
          inc_fixed_2.5                         1.042       2.500  
          wls_recency_h6                        0.840      10.857  
          wls_recency_h6_inc_fixed_2.5          1.030       2.500  
2022      ols                                   0.817      11.718  
          inc_fixed_2.5                         1.023       2.500  
          wls_recency_h6                        0.853       9.109  
          wls_recency_h6_inc_fixed_2.5          0.999       2.500

In [9]:
# each model's coefficients averaged over the windows
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

,Intercept,gen_margin_raw,partisan_lean,inc_signed
run,,,,
ols,-0.228,0.865,0.811,12.776
inc_fixed_2.5,-0.215,0.791,1.044,2.500
wls_recency_h6_inc_fixed_2.5,-0.692,0.836,1.034,2.500
wls_recency_h6,-0.497,0.889,0.844,10.831
